Refactored de-warping script (uses precomputed run_manifests).

[Runtime: ~1 min per scan file]

----------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS & SET COMPUTATIONAL PARAMETERS:

import os, sys, csv, json, math, glob, re, time
import psutil
import subprocess
from datetime import datetime
import tempfile
import shutil

# Note: these environmental parameters need to be set PRIOR to certain specific library imports, hence its placement here:
try:
    cores = psutil.cpu_count(logical=False) or os.cpu_count() or 2
except Exception:
    cores = os.cpu_count() or 2
os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = str(cores)
os.environ["OMP_NUM_THREADS"] = str(cores)
os.environ["MKL_NUM_THREADS"] = str(cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(cores)
print(f"[Parallelization:] Planning to use {cores} processing threads.")

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
from scipy.ndimage import gaussian_filter, gaussian_gradient_magnitude, binary_dilation, binary_erosion
from scipy.ndimage import distance_transform_edt as _dt
import ants
try:
    from skimage.feature import canny as sk_canny
    HAVE_SK = True
except Exception:
    HAVE_SK = False

In [ ]:
### LOAD FREESURFER:
FREESURFER_HOME = config['freesurfer']['home']
FREESURFER_LICENSE = Path(config['freesurfer'].get('license'))
FREESURFER_SUBJECTS_DIR = config['freesurfer']['subjects_dir']
os.environ['FREESURFER_HOME'] = str(FREESURFER_HOME)
os.environ['SUBJECTS_DIR']    = str(FREESURFER_SUBJECTS_DIR)
os.environ['PATH'] = f"{str(FREESURFER_HOME)}/bin:" + os.environ.get('PATH', '')
if FREESURFER_LICENSE.is_file():
    os.environ['FS_LICENSE'] = str(FREESURFER_LICENSE)
else:
    raise RuntimeError("FreeSurfer license not found: check that 'license.txt' exists in home FreeSurfer directory."
    "in FreeSurfer $HOME directory and that a valid path is set in 'config.yaml' file.")
subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

# Optional filtering params:
FILTER_SUBJECT_IDS = config["filter"].get("subject_IDs", [])
FILTER_SESSION_IDS = config["filter"].get("session_IDs", [])
FILTER_GROUP_IDS   = config["filter"].get("group_IDs", [])

OVERWRITE_SDC = config['overwrite_SDC']

# Processing parameters:

SDC_REG_ITERS = tuple(config['SDC_parameters']['SDC_num_iterations'])
SDC_USE_SAFE_CROP = config['SDC_parameters']['crop_to_T1_mask']
SDC_SMOOTH_SIGMA_VOX = config['SDC_parameters']['gaussian_smoothing']
if type(config['SDC_parameters']['downsample_T1']) == list:
    if len(config['SDC_parameters']['downsample_T1']) == 3:
        SDC_FIXED_DOWNSAMPLE_MM = tuple(config['SDC_parameters']['downsample_T1'])
    else:
        raise Exception("The 'downsample_T1' parameter must either be 'None' OR be provided as a 3-item list/tuple; check CONFIG settings.")
else:
    SDC_FIXED_DOWNSAMPLE_MM = False        # 'False' => native T1 resolution (default); setting as tuple downsamples T1 images down to that isotropic resolution (e.g. '(1.5, 1.5, 1.5)'), makes processing simpler/faster but not otherwise recommended


#############################################################################
# QC output-related parameters (Not exposed to users, back-end only):
QC_BLEND_ALPHA       = 0.30   # tint strength
QC_CANNY_SIGMA       = 1.2    # Canny blur, in voxels (only used if 'scikit-image' is available)
QC_OUTLINE_TOUCH_R   = 1      # how close an EPI edge must be to count as “touching” outline (pixels)
QC_CROP_PAD          = 3
QC_INT_PLOTLIMS      = (2, 98)
DPI                  = 170
FIGSIZE              = (18, 6)
#############################################################################


# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config['root_output_directory'])

### INPUTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
fMRI_DATA_DIR = config['fMRI_data_directory']
fMRI_PARAMETERS_PATH = Path(ROOT_DIR) / 'fMRI_manifest.csv'

BOLDREF_DIR = Path(ROOT_DIR) / config['EPI_ref_dir']
MASK_DIR = Path(ROOT_DIR) / config['motion_mask_dir']
TRANSFORMS_DIR = Path(ROOT_DIR) / config['motion_xforms_dir']
CONFOUNDS_DIR = Path(ROOT_DIR) / config['confounds_dir']
FULL_MC_VOLUME_DIR = Path(ROOT_DIR) / config['motion_correction_output_dir']

# OUTPUTS:
SDC_OUTPUT_DIR = Path(ROOT_DIR) / config['SDC_output_dir']
SDC_QC_SUBDIR = Path(ROOT_DIR) / config['SDC_QC_subdir']


# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs = pd.read_csv(fMRI_PARAMETERS_PATH)

In [ ]:
# =====================================================================
# FILTERING (group_ID → session_ID → subject_ID) + DIAGNOSTIC SUBSETTING
# =====================================================================

# ---- FILTERING (if enabled) ----
any_filters_active = bool(FILTER_GROUP_IDS or FILTER_SESSION_IDS or FILTER_SUBJECT_IDS)
if any_filters_active:
    print("\n[FILTER] Applying YAML-defined filters to fMRI_runs...")
    print(f"[FILTER] Starting with {len(fMRI_runs):,} rows.")
    # 1) Filter by group_IDs (substrings, case-insensitive):
    if FILTER_GROUP_IDS:
        n_before = len(fMRI_runs)
        pattern = "|".join(re.escape(val) for val in FILTER_GROUP_IDS)
        mask = fMRI_runs["group_ID"].astype(str).str.contains(pattern, case=False, na=False)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] group_IDs {FILTER_GROUP_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 2) Filter by session_IDs (exact matches):
    if FILTER_SESSION_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["session_ID"].astype(str).isin(FILTER_SESSION_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] session_IDs {FILTER_SESSION_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    # 3) Filter by subject_IDs (exact matches):
    if FILTER_SUBJECT_IDS:
        n_before = len(fMRI_runs)
        mask = fMRI_runs["subject_ID"].astype(str).isin(FILTER_SUBJECT_IDS)
        fMRI_runs = fMRI_runs.loc[mask].copy()
        print(f"[FILTER] subject_IDs {FILTER_SUBJECT_IDS} → {mask.sum():,} of {n_before:,} rows kept.")
    print(f"[FILTER] Final row count after all filters: {len(fMRI_runs):,} rows.\n")

# ---- DIAGNOSTIC SUBSETTING (if enabled) ----
if isinstance(SUBSET, int) and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    fMRI_runs = fMRI_runs.head(SUBSET).copy()

if any_filters_active or SUBSET:
    fMRI_runs = fMRI_runs.reset_index(drop=True)
    display(fMRI_runs)

---------
First step: prep anatomy & other input variables / filepaths / etc.

In [ ]:
# =========================
# SDC — Step 1: input audit + FS anatomy export prep
# =========================

# Set "prefix" ('<subject_ID>_<session_ID>_*' format):
if "prefix" not in fMRI_runs.columns:
    fMRI_runs["prefix"] = fMRI_runs["subject_ID"].astype(str) + "_" + fMRI_runs["session_ID"].astype(str)

# Resolve expected inputs (no recursive glob; use new naming convention)
def _exists(p): 
    try: return Path(p).exists()
    except: return False

boldref_paths   = []
mask_paths      = []
conf_tsv_paths  = []
conf_json_paths = []
motion_counts   = []
motion_sample   = []
fs_dirs         = []
t1_brain_mgz    = []
t1_mask_mgz     = []
has_fs_brain    = []
has_fs_mask     = []

issues = []

for r in fMRI_runs.itertuples(index=False):
    subj = str(r.subject_ID); sess = str(r.session_ID); pref = f"{subj}_{sess}"

    boldref_p = Path(BOLDREF_DIR) / f"{pref}_boldref.nii"
    mask_p    = Path(MASK_DIR)    / f"{pref}_boldref_mask.nii"
    conf_tsv  = Path(CONFOUNDS_DIR) / f"{pref}_confounds.tsv"
    conf_json = Path(CONFOUNDS_DIR) / f"{pref}_confounds.json"

    # Motion transforms live in subdirectories named by prefix:
    xdir      = Path(TRANSFORMS_DIR) / pref
    mc_list   = sorted(glob.glob(str(xdir / "mc*")) ) if xdir.is_dir() else []
    mc_n      = len(mc_list)
    mc_one    = mc_list[0] if mc_list else ""

    # FreeSurfer anatomy data objects:
    fs_dir    = Path(FREESURFER_SUBJECTS_DIR) / subj
    brain_mgz = fs_dir / "mri" / "brain.mgz"
    bmask_mgz = fs_dir / "mri" / "brainmask.mgz"
    has_b     = brain_mgz.exists()
    has_m     = bmask_mgz.exists()

    # Record into arrays:
    boldref_paths.append(str(boldref_p))
    mask_paths.append(str(mask_p))
    conf_tsv_paths.append(str(conf_tsv))
    conf_json_paths.append(str(conf_json))
    motion_counts.append(mc_n)
    motion_sample.append(mc_one)
    fs_dirs.append(str(fs_dir))
    t1_brain_mgz.append(str(brain_mgz))
    t1_mask_mgz.append(str(bmask_mgz))
    has_fs_brain.append(bool(has_b))
    has_fs_mask.append(bool(has_m))

    # Collect issues (only if something is missing/problematic):
    missing_bits = []
    if not boldref_p.exists(): missing_bits.append("boldref")
    if not mask_p.exists():    missing_bits.append("mask")
    if not has_b:              missing_bits.append("FS brain.mgz")
    if not has_m:              missing_bits.append("FS brainmask.mgz")
    if missing_bits:
        issues.append({"subject_ID": subj, "session_ID": sess,
                       "issue": "missing: " + ", ".join(missing_bits)})

# augment 'fMRI_runs' in-place:
fMRI_runs["boldref_path"]   = boldref_paths
fMRI_runs["mask_path"]      = mask_paths
fMRI_runs["confounds_tsv"]  = conf_tsv_paths
fMRI_runs["confounds_json"] = conf_json_paths
fMRI_runs["motion_n_files"] = motion_counts
fMRI_runs["has_motion"]     = [n > 0 for n in motion_counts]
fMRI_runs["motion_sample"]  = motion_sample
fMRI_runs["fs_subject_dir"] = fs_dirs
fMRI_runs["t1_brain_mgz"]   = t1_brain_mgz
fMRI_runs["t1_mask_mgz"]    = t1_mask_mgz
fMRI_runs["has_fs_brain"]   = has_fs_brain
fMRI_runs["has_fs_mask"]    = has_fs_mask

def _exists(p):
    try:
        return Path(p).exists()
    except Exception:
        return False

# Set auditing flags:
fMRI_runs["has_boldref"]   = fMRI_runs["boldref_path"].apply(_exists)
fMRI_runs["has_mask"]      = fMRI_runs["mask_path"].apply(_exists)
fMRI_runs["has_confounds"] = fMRI_runs["confounds_tsv"].apply(_exists)

fMRI_runs["ready_for_SDC"] = (
    fMRI_runs["has_boldref"] &
    fMRI_runs["has_mask"] &
    fMRI_runs["has_fs_brain"] &
    fMRI_runs["has_fs_mask"])

# Otional issues dataframe (only if problems arise):
SDC_filecheck_issues = (
    pd.DataFrame(issues)
    if len(issues)
    else pd.DataFrame(columns=["subject_ID","session_ID","issue"]))

# Print quick console summary:
n_total = len(fMRI_runs)
n_ready = int(fMRI_runs["ready_for_SDC"].sum())
n_missing_masks = int((~fMRI_runs["has_mask"]).sum())
n_missing_fs = int((~(fMRI_runs["has_fs_brain"] & fMRI_runs["has_fs_mask"])).sum())
n_missing_ped = int(
    (fMRI_runs["PED_axis"].astype(str).str.strip() == "").sum()) if "PED_axis" in fMRI_runs.columns else 0

print(f"[SDC] Runs indexed: {n_total}")
print(f"[SDC] Ready for SDC now (boldref+mask+FS anatomy): {n_ready}/{n_total}")
print(f"[SDC] Missing masks: {n_missing_masks}")
print(f"[SDC] Missing FS anatomy (brain.mgz and/or brainmask.mgz): {n_missing_fs}")
print(f"[SDC] PED_axis unknown: {n_missing_ped} (non-blocking for now)")

if not SDC_filecheck_issues.empty:
    print("\n[SDC] Example issues:")
    display(SDC_filecheck_issues.head(10))

# with pd.option_context("display.max_colwidth", 140):
#     display(fMRI_runs.head(10)[[
#         "prefix","subject_ID","session_ID","has_boldref","has_mask","has_confounds","has_motion",
#         "has_fs_brain","has_fs_mask","PED_axis","RepetitionTime","TotalReadoutTime",
#         "boldref_path","mask_path","confounds_tsv","motion_sample","fs_subject_dir"]])

# ============================================================
# Step 2 — FreeSurfer anatomy export (w/ optional caching) + input validation
# ============================================================

# Toggles (to persist cache -- always-on; not exposed to users):
CACHE_FS_NIFTI = True
ANAT_CACHE_DIR = Path(SDC_OUTPUT_DIR) / "_anat_cache"
ANAT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FS_SETUP = Path(FREESURFER_HOME) / "SetUpFreeSurfer.sh"
if not FS_SETUP.exists():
    raise RuntimeError(f"FreeSurfer setup script not found at: {FS_SETUP}")

def run_bash(cmd, env=None):
    r = subprocess.run(["bash", "-lc", cmd],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                       text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(f"CMD failed ({r.returncode}): {cmd}\n"
                           f"STDOUT:\n{r.stdout}\nSTDERR:\n{r.stderr}")
    return r.stdout

def export_fs_to_nii(fs_subject_dir, prefix, fs_setup=str(FS_SETUP)):
    """
    Convert FreeSurfer brain.mgz and brainmask.mgz to uncompressed NIfTI (.nii)
    for ANTs. If CACHE_FS_NIFTI=True, writes to a flat cache dir with
    <prefix>_T1_brain.nii and <prefix>_T1_brain_mask.nii; else temp dir.
    """
    mgz_brain = Path(fs_subject_dir) / "mri" / "brain.mgz"
    mgz_mask  = Path(fs_subject_dir) / "mri" / "brainmask.mgz"
    if not (mgz_brain.exists() and mgz_mask.exists()):
        raise FileNotFoundError(f"Missing FS anatomy: {mgz_brain} or {mgz_mask}")

    if CACHE_FS_NIFTI:
        out_dir = ANAT_CACHE_DIR
        cleanup_fn = None
        t1_nii  = out_dir / f"{prefix}_T1_brain.nii"
        t1m_nii = out_dir / f"{prefix}_T1_brain_mask.nii"
    else:
        out_dir = Path(tempfile.mkdtemp(prefix=f"sdc_anat_{prefix}_"))
        def _cleanup():
            try: shutil.rmtree(out_dir)
            except Exception: pass
        cleanup_fn = _cleanup
        t1_nii  = out_dir / f"{prefix}_T1_brain.nii"
        t1m_nii = out_dir / f"{prefix}_T1_brain_mask.nii"

    def _convert(mgz_in, nii_out):
        if Path(nii_out).exists():
            return
        try:
            cmd = f"source '{fs_setup}'; mri_convert '{mgz_in}' '{nii_out}'"
            run_bash(cmd)
        except Exception as e:
            print("[WARN] mri_convert failed; falling back to nibabel:", e)
            img = nib.load(str(mgz_in))
            nib.save(nib.Nifti1Image(img.get_fdata(), img.affine, img.header), str(nii_out))

    _convert(mgz_brain, t1_nii)
    _convert(mgz_mask,  t1m_nii)
    return str(t1_nii), str(t1m_nii), cleanup_fn

def affine_close(a, b, rtol=1e-4, atol=1e-3):
    return np.allclose(a, b, rtol=rtol, atol=atol)

def validate_run_inputs(boldref_path, mask_path, t1_path, t1mask_path):
    br = nib.load(boldref_path); br_data = br.get_fdata()
    mk = nib.load(mask_path);    mk_data = mk.get_fdata()
    t1 = nib.load(t1_path);      tm = nib.load(t1mask_path)

    ok_shapes = br.shape[:3] == mk.shape[:3]
    ok_aff    = affine_close(br.affine, mk.affine)
    if not ok_shapes or not ok_aff:
        print("[FAIL] boldref vs mask mismatch:",
              f"shape {br.shape[:3]} vs {mk.shape[:3]} | affine_close={ok_aff}")
    else:
        vox_mm = tuple(np.round(nib.affines.voxel_sizes(br.affine), 3))
        print(f"[OK] boldref & mask share grid: shape={br.shape[:3]}, vox_mm={vox_mm}")

    t1_ok_shapes = t1.shape[:3] == tm.shape[:3]
    t1_ok_aff    = affine_close(t1.affine, tm.affine)
    if not t1_ok_shapes or not t1_ok_aff:
        print("[FAIL] T1 vs T1_mask mismatch:",
              f"shape {t1.shape[:3]} vs {tm.shape[:3]} | affine_close={t1_ok_aff}")
    else:
        t1_vox_mm = tuple(np.round(nib.affines.voxel_sizes(t1.affine), 3))
        print(f"[OK] T1 brain & mask share grid: shape={t1.shape[:3]}, vox_mm={t1_vox_mm}")

    mk_bin = (mk_data > 0).astype(np.uint8)
    cov = mk_bin.sum() / np.prod(mk_bin.shape)
    note = ""
    if cov < 0.10: note = " (low coverage?)"
    if cov > 0.40: note = " (very high coverage?)"
    print(f"[MASK] EPI brain mask coverage: {cov:.3f}{note}")

def prepare_runs(fmri_df, n=None):
    """
    For the first n runs with ready_for_SDC=True, export FS anatomy to NIfTI
    and validate inputs. Returns a prep_manifest (like old code).
    """
    ready = fmri_df[fmri_df["ready_for_SDC"]].copy()
    if len(ready) == 0:
        raise RuntimeError("No runs ready_for_SDC found in fMRI_runs.")
    if isinstance(SUBSET, int) and SUBSET > 0:
        ready = ready.head(SUBSET).copy()
    elif n is not None:
        ready = ready.head(n).copy()

    prepped = []
    for _, row in ready.iterrows():
        pref = row["prefix"]; subj = row["subject_ID"]; sess = row["session_ID"]
        print(f"\n=== Prep: {pref} ===")
        # Export FS anatomy
        t1_nii, t1m_nii, cleanup = export_fs_to_nii(row["fs_subject_dir"], pref)
        # Validate
        validate_run_inputs(row["boldref_path"], row["mask_path"], t1_nii, t1m_nii)
        prepped.append({
            "prefix": pref,
            "subject_ID": subj,
            "session_ID": sess,
            "boldref_path": row["boldref_path"],
            "mask_path": row["mask_path"],
            "t1_brain_nii": t1_nii,
            "t1_mask_nii": t1m_nii,
            "PED_axis": row.get("PED_axis",""),
            "RepetitionTime": row.get("RepetitionTime",""),
            "TotalReadoutTime": row.get("TotalReadoutTime",""),
            "_cleanup_fn": cleanup})
    return pd.DataFrame(prepped)

# Prepare runs:
prep_manifest = prepare_runs(fMRI_runs)

# print("\n[PREP] Prepared runs:")
# with pd.option_context("display.max_colwidth", 160):
#     display(prep_manifest[[
#         "prefix","subject_ID","session_ID",
#         "boldref_path","mask_path","t1_brain_nii","t1_mask_nii","PED_axis","RepetitionTime","TotalReadoutTime"]])

print(f"\n[CACHE] CACHE_FS_NIFTI={CACHE_FS_NIFTI} | cache_dir='{ANAT_CACHE_DIR}'")

-----------
All data-auditing prepped; now perform actual SDC:

In [ ]:
# ==========================================================
# STEP 3 — SDC-SyN (ANTsPy, PRODUCTION); uses full-resolution T1 data
#   * Fieldmap-less SyN, inverse warp in EPI grid
#   * Axis-constrain to PED axis (+ smoothing)
#   * Displacement stats + QC PNG + provenance
#   * 4-level schedule; threads configurable
#   * Keeps mask-based crop of T1 (fast + safe)
# ==========================================================

# Re-use thread count chosen during initialization:
SDC_THREADS = cores
try:
    ants.set_number_of_threads(SDC_THREADS)
except Exception:
    pass
print(f"[SDC] Using {SDC_THREADS} threads")

print("[SDC] Schedule:", SDC_REG_ITERS)

# Set QC destination directory for SDC (from config/init):
QC_DIR = SDC_QC_SUBDIR
QC_DIR.mkdir(parents=True, exist_ok=True)
SDC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SDC_MAX_ABS_DISP_WARN_MM = 12.0   # <-- warn if '|max displacement|' exceeds this
SDC_MIN_DELTA_NCC = 0.005

# Define helper functions:

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def ped_axis_to_index(s):
    s = (s or "").strip().lower()
    if s not in ("i", "j", "k"):
        raise ValueError(f"PED_axis must be one of 'i','j','k'; got {s!r}")
    return {"i": 0, "j": 1, "k": 2}[s]

def pick_inverse_warp(lst):
    if not lst:
        return None
    for p in lst:
        low = p.lower()
        if "inversewarp" in low and (low.endswith(".nii") or low.endswith(".nii.gz")):
            return p
    for p in lst:
        low = p.lower()
        if "warp" in low and (low.endswith(".nii") or low.endswith(".nii.gz")):
            return p
    return None

def binarize_mask(mask_img, thresh=0.5):
    arr = mask_img.numpy()
    bin_arr = (arr > thresh).astype(np.uint8)
    return ants.from_numpy(
        bin_arr,
        origin=mask_img.origin,
        spacing=mask_img.spacing,
        direction=mask_img.direction)

def safe_crop(image_img, mask_img):
    if not np.any(mask_img.numpy() > 0):
        return image_img, mask_img, False
    try:
        img_cr  = ants.crop_image(image_img, mask_img, label=1)
        mask_cr = ants.crop_image(mask_img,  mask_img,  label=1)
        return img_cr, mask_cr, True
    except Exception:
        return image_img, mask_img, False

def extract_kept_component_as_image(vector_img, keep_i):
    vf = vector_img.numpy().astype(np.float32)   # <-- dimensionality = (X,Y,Z,3)
    if vf.ndim != 4 or vf.shape[3] != 3:
        raise RuntimeError(f"Unexpected vector field shape {vf.shape}")
    kept = vf[..., keep_i]                       # <-- dimensionality = (X,Y,Z)
    return ants.from_numpy(
        kept,
        origin=vector_img.origin,
        spacing=vector_img.spacing,
        direction=vector_img.direction)

def disp_stats_and_png(ax_warp_nii, epi_mask_img, ped_axis, disp_png):
    keep_i = ped_axis_to_index(ped_axis)
    vec_img  = ants.image_read(ax_warp_nii)                    # <-- vector field (X,Y,Z,3)
    kept_img = extract_kept_component_as_image(vec_img, keep_i)

    # Smooth the kept component(s) (already done in build step too, but harmless to include here):
    if SDC_SMOOTH_SIGMA_VOX and SDC_SMOOTH_SIGMA_VOX > 0:
        kept_np = gaussian_filter(
            kept_img.numpy(),
            sigma=SDC_SMOOTH_SIGMA_VOX,
            mode="nearest")
        kept_img = ants.from_numpy(
            kept_np,
            origin=kept_img.origin,
            spacing=kept_img.spacing,
            direction=kept_img.direction)

    # Align grids for stats:
    if kept_img.shape != epi_mask_img.shape:
        kept_img = ants.resample_image_to_target(kept_img, epi_mask_img, interp_type=0)

    kept_np = kept_img.numpy()
    mask_np = (epi_mask_img.numpy() > 0)
    absd  = np.abs(kept_np[mask_np])
    dmean = float(absd.mean()) if absd.size else float("nan")
    dp95  = float(np.percentile(absd, 95)) if absd.size else float("nan")
    dmax  = float(absd.max()) if absd.size else float("nan")

    # Render with symmetric limits around 0 for readability:
    mid = kept_np.shape[2] // 2
    vmax = np.percentile(np.abs(kept_np[mask_np]), 99) if absd.size else np.nan

    plt.figure(figsize=(5, 4))
    im = plt.imshow(
        kept_np[:, :, mid].T,
        origin="lower",
        vmin=-vmax,
        vmax=+vmax,
        cmap="coolwarm")
    plt.colorbar(im, label=f"PE-axis disp (mm, '{ped_axis}')")
    plt.title(f"{os.path.basename(os.path.dirname(ax_warp_nii))}: mid-axial PE displacement")
    plt.tight_layout()
    plt.savefig(disp_png, dpi=150)
    plt.close()

    warn = (
        "max|disp|={:.1f}mm > {:.1f}mm".format(dmax, SDC_MAX_ABS_DISP_WARN_MM)
        if (not np.isnan(dmax) and dmax > SDC_MAX_ABS_DISP_WARN_MM)
        else "")
    return dmean, dp95, dmax, warn

def prepare_fixed_and_mask(t1_img, t1_mask_img):
    """Return (fixed, fixed_mask) after optional downsample + binarize + crop."""
    if SDC_FIXED_DOWNSAMPLE_MM:  # e.g., (1.5,1.5,1.5)
        fixed      = ants.resample_image(t1_img, SDC_FIXED_DOWNSAMPLE_MM, use_voxels=False, interp_type=0)
        fixed_mask = ants.resample_image_to_target(t1_mask_img, fixed, interp_type=1)
    else:
        fixed, fixed_mask = t1_img, t1_mask_img

    fixed_mask = binarize_mask(fixed_mask, 0.5)
    if SDC_USE_SAFE_CROP:
        fixed, fixed_mask, _ = safe_crop(fixed, fixed_mask)
    return fixed, fixed_mask

def sdc_syn_estimate_for_run(row):
    """
    Estimate SDC inverse warp in EPI grid (axis-constrained), compute stats, save QC.
    Returns dict with metrics and file paths.
    """
    prefix       = row["prefix"]   # <-- now a single compound identifier '{<subject_ID>_<session_ID>}'
    boldref_path = row["boldref_path"]
    mask_path    = row["mask_path"]
    t1_path      = row["t1_brain_nii"]
    t1mask_path  = row["t1_mask_nii"]
    ped_axis     = row["PED_axis"]

    if not ped_axis:
        raise RuntimeError(f"{prefix}: missing PED_axis.")

    # Outputs:
    run_out = Path(SDC_OUTPUT_DIR) / prefix
    ensure_dir(run_out)
    raw_invwarp_nii = str(run_out / "sdc_invwarp_raw.nii")
    ax_warp_nii     = str(run_out / "sdc_warp_axcon.nii")
    disp_png        = str(QC_DIR / f"{prefix}_03_sdc_disp.png")
    prov_json       = str(run_out / "sdc_provenance.json")

    # If warp exists and not overwriting, reuse stats/PNG and return:
    if (not OVERWRITE_SDC) and os.path.exists(ax_warp_nii) and os.path.exists(raw_invwarp_nii):
        epi_mask_img = ants.image_read(mask_path)
        dmean, dp95, dmax, warn = disp_stats_and_png(ax_warp_nii, epi_mask_img, ped_axis, disp_png)
        return {
            "prefix": prefix,
            "axis": ped_axis,
            "disp_mean_mm": dmean,
            "disp_p95_mm": dp95,
            "disp_max_mm": dmax,
            "raw_warp": raw_invwarp_nii,
            "axis_constrained_warp": ax_warp_nii,
            "disp_png": disp_png,
            "warn": warn}

    t0 = time.time()

    # Load images:
    fixed0      = ants.image_read(t1_path)      # <-- T1 brain (in native resolution)
    fixed_mask0 = ants.image_read(t1mask_path)  # <-- T1 mask
    moving      = ants.image_read(boldref_path) # <-- EPI boldref (i.e. provides the target grid for warp)
    moving_mask = ants.image_read(mask_path)

    # Prep fixed (optionally downsample, then binarize+crop):
    fixed, fixed_mask = prepare_fixed_and_mask(fixed0, fixed_mask0)

    # Registration (4-level SyN); metrics chosen for robustness (Mattes/MI):
    tx = ants.registration(
        fixed=fixed,
        moving=moving,
        type_of_transform="SyN",
        mask=fixed_mask,
        moving_mask=moving_mask,
        regIterations=tuple(SDC_REG_ITERS),
        aff_metric="mattes",
        syn_metric="mattes",
        outprefix=str(run_out / "syn_"),
        verbose=False)

    inv_warp_path = pick_inverse_warp(tx.get("invtransforms", []))
    if inv_warp_path is None:
        raise RuntimeError(
            f"{prefix}: inverse warp not found in invtransforms={tx.get('invtransforms', [])}")

    inv_warp_img = ants.image_read(inv_warp_path)     # <-- vector image, usually EPI grid
    ants.image_write(inv_warp_img, raw_invwarp_nii)   # <-- uncompressed copy (stable filename)

    # Axis-constrain the inverse warp (keep only PE component; and smooth):
    keep_i = ped_axis_to_index(ped_axis)
    vf_np  = inv_warp_img.numpy().astype(np.float32)  # <-- dimensionality = (X,Y,Z,3)
    kept   = vf_np[..., keep_i].copy()
    if SDC_SMOOTH_SIGMA_VOX and SDC_SMOOTH_SIGMA_VOX > 0:
        kept = gaussian_filter(kept, sigma=SDC_SMOOTH_SIGMA_VOX, mode="nearest")
    vf_ax = np.zeros_like(vf_np, dtype=np.float32)
    vf_ax[..., keep_i] = kept
    vf_ax_img = ants.from_numpy(
        vf_ax,
        origin=inv_warp_img.origin,
        spacing=inv_warp_img.spacing,
        direction=inv_warp_img.direction,
        has_components=True)
    ants.image_write(vf_ax_img, ax_warp_nii)

    # Compute stats & write output PNG image:
    epi_mask_img = (
        moving_mask
        if moving_mask.shape == inv_warp_img.shape[:3]
        else ants.resample_image_to_target(moving_mask, inv_warp_img, interp_type=1))
    dmean, dp95, dmax, warn = disp_stats_and_png(ax_warp_nii, epi_mask_img, ped_axis, disp_png)

    # Write provenance logs:
    with open(prov_json, "w") as f:
        json.dump(
            {
                "prefix": prefix,
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "inputs": {
                    "boldref": boldref_path,
                    "boldref_mask": mask_path,
                    "t1_brain": t1_path,
                    "t1_brain_mask": t1mask_path,
                    "PED_axis": ped_axis,
                    "RepetitionTime": row.get("RepetitionTime", ""),
                    "TotalReadoutTime": row.get("TotalReadoutTime", "")},
                "outputs": {
                    "raw_inverse_warp": raw_invwarp_nii,
                    "axis_constrained_warp": ax_warp_nii,
                    "disp_png": disp_png},
                "params": {
                    "SDC_SMOOTH_SIGMA_VOX": SDC_SMOOTH_SIGMA_VOX,
                    "regIterations": list(SDC_REG_ITERS),
                    "fixed_downsample_mm": (
                        list(SDC_FIXED_DOWNSAMPLE_MM) if SDC_FIXED_DOWNSAMPLE_MM else None),
                    "fixed_cropped": bool(SDC_USE_SAFE_CROP),
                    "threads": SDC_THREADS,
                    "SDC_MIN_DELTA_NCC": SDC_MIN_DELTA_NCC,
                    "SDC_MAX_ABS_DISP_WARN_MM": SDC_MAX_ABS_DISP_WARN_MM},
                "software": {
                    "antspyx_version": getattr(ants, "__version__", "unknown")},},f,indent=2)

    elapsed = time.time() - t0
    return {
        "prefix": prefix,
        "axis": ped_axis,
        "disp_mean_mm": dmean,
        "disp_p95_mm": dp95,
        "disp_max_mm": dmax,
        "raw_warp": raw_invwarp_nii,
        "axis_constrained_warp": ax_warp_nii,
        "disp_png": disp_png,
        "warn": warn,
        "elapsed_s": elapsed}

# _______________________________________________________________________________________________________________________
# Main execution:

rows = prep_manifest.to_dict(orient="records")

results = []
for row in rows:
    try:
        print(f"\n[SDC] Starting {row['prefix']} (axis={row['PED_axis']})")
        res = sdc_syn_estimate_for_run(row)
        results.append(res)
        print(
            f"[SDC OK] {res['prefix']}: "
            f"mean={res['disp_mean_mm']:.2f} | p95={res['disp_p95_mm']:.2f} | max={res['disp_max_mm']:.2f} mm")
        if res["warn"]:
            print(f"         [WARN] {res['warn']}")
        print(f"         raw: {res['raw_warp']}")
        print(f"         ax : {res['axis_constrained_warp']}")
        print(f"         png: {res['disp_png']}")
    except Exception as e:
        print(f"[SDC FAIL] {row.get('prefix','<unknown>')}: {e}")

if results:
    df = pd.DataFrame(results)
    with pd.option_context("display.max_colwidth", 160):
        display(df[[
            "prefix", "axis",
            "disp_mean_mm", "disp_p95_mm", "disp_max_mm",
            "warn", "axis_constrained_warp"]])
else:
    print("[SDC] No results to show.")

In [ ]:
# ==========================================================
# STEP 4 — APPLY SDC (DE-WARPING ONLY):
#   * Reads axis-constrained inverse warp from Step 3
#   * Applies it to the BOLD reference in-place grid
#   * Writes: <SDC_OUTPUT_DIR>/<prefix>/boldref_sdc.nii
#   * No QC or metrics here (see later cells)
# ==========================================================

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def apply_sdc_for_run(row):
    """
    Apply the axis-constrained inverse warp to the EPI boldref, producing boldref_sdc.

    Returns the output path.
    """
    prefix       = row["prefix"]
    boldref_path = row["boldref_path"]  # <-- original (distorted) EPI ref

    run_out = Path(SDC_OUTPUT_DIR) / prefix
    ensure_dir(run_out)

    # Inputs made by Step 3
    ax_warp_nii = run_out / "sdc_warp_axcon.nii"
    if not ax_warp_nii.exists():
        raise FileNotFoundError(f"{prefix}: missing SDC warp {ax_warp_nii} (run Step 3 first)")

    # Output
    out_sdc = run_out / "boldref_sdc.nii"
    if (not OVERWRITE_SDC) and out_sdc.exists():
        print(f"[APPLY SKIP] {prefix}: exists and OVERWRITE_SDC=False → {out_sdc}")
        return str(out_sdc)

    # Apply displacement field to the BOLD reference (staying on the EPI grid):
    moving = ants.image_read(str(boldref_path))  # <-- distorted EPI ref (moving)
    epi_sdc = ants.apply_transforms(
        fixed=moving,     # <-- keep native EPI voxel grid
        moving=moving,
        transformlist=[str(ax_warp_nii)],
        interpolator="linear")
    ants.image_write(epi_sdc, str(out_sdc))

    print(f"[APPLY OK] {prefix}: wrote {out_sdc}")
    return str(out_sdc)

# ____________________________________________________________________________________________________________
# Main execution:

rows = prep_manifest.to_dict(orient="records")

applied = []
for row in rows:
    try:
        p = apply_sdc_for_run(row)
        applied.append({"prefix": row["prefix"], "boldref_sdc": p})
    except Exception as e:
        print(f"[APPLY FAIL] {row.get('prefix','<unknown>')}: {e}")

if applied:
    df = pd.DataFrame(applied)
    with pd.option_context("display.max_colwidth", 160):
        display(df)
else:
    print("[APPLY] No runs processed.")

In [ ]:
# ==========================================================
# STEP 5 — APPLY SDC TO FULL 4D:
#   Inputs (must exist):
#     • Motion-corrected 4D:  '<FULL_MC_VOLUME_DIR>/<prefix>_bold_mc.nii'
#     • Axis-constrained inv warp: '<SDC_OUTPUT_DIR>/<prefix>/sdc_warp_axcon.nii'
#     • Undistorted reference:    '<SDC_OUTPUT_DIR>/<prefix>/boldref_sdc.nii'
#     • Distorted BOLDREF (for geometry template): row['boldref_path']
#   Output:
#     • '<SDC_OUTPUT_DIR>/<prefix>/bold_mc_sdc.nii.gz' (4D, undistorted lattice)
# ==========================================================

def _ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def _require_file(path_str, err_tag):
    if (not path_str) or (not os.path.exists(path_str)):
        raise FileNotFoundError(f"{err_tag} not found: {path_str}")
    return path_str

def _apply_sdc_to_known_mc4d(row, overwrite=False):
    """
    row must include:
      - prefix
      - boldref_path (distorted BOLDREF used for SDC; geometry template for per-frame images)
    Globals expected:
      - FULL_MC_VOLUME_DIR, SDC_OUTPUT_DIR
    """
    prefix = str(row["prefix"])

    # Load up motion-corrected 4D from previous stage:
    mc4d_path = Path(FULL_MC_VOLUME_DIR) / f"{prefix}_bold_mc.nii"
    run_out   = Path(SDC_OUTPUT_DIR) / prefix
    _ensure_dir(run_out)

    ax_warp = run_out / "sdc_warp_axcon.nii"
    ref_sdc = run_out / "boldref_sdc.nii"
    out_4d  = run_out / "bold_mc_sdc.nii.gz"

    # Sanity-check -- require everything explicitly:
    _require_file(str(mc4d_path),  f"[{prefix}] motion-corrected 4D (FULL_MC_VOLUME_DIR)")
    _require_file(str(ax_warp),    f"[{prefix}] SDC axis-constrained inverse warp")
    _require_file(str(ref_sdc),    f"[{prefix}] SDC undistorted reference")
    moving_ref_path = _require_file(
        str(row.get("boldref_path", "")),
        f"[{prefix}] distorted BOLDREF (prep_manifest['boldref_path'])")

    if (not overwrite) and out_4d.exists():
        print(f"[SDC-4D SKIP] {prefix}: exists and overwrite=False → {out_4d}")
        return str(out_4d)

    # Load fixed (undistorted lattice) and moving-geometry template (distorted):
    fixed_img_ants  = ants.image_read(str(ref_sdc))          # <-- ANTs image (target lattice)
    fixed_ref_nib   = nib.load(str(ref_sdc))                 # <-- NIfTI for affine/spacing
    moving_ref_ants = ants.image_read(moving_ref_path)       # <-- distorted geometry, per-frame

    # Load MC 4D with nibabel; keep its timing info (TR):
    mc4d_nib = nib.load(str(mc4d_path))
    data     = mc4d_nib.get_fdata(dtype=np.float32)
    if data.ndim != 4:
        raise ValueError(f"[{prefix}] Motion-corrected path is not 4D: {mc4d_path}")
    X, Y, Z, T = data.shape

    # Get TR from MC header (if present), & spacing (from fixed reference):
    mc_zooms = mc4d_nib.header.get_zooms()
    TR = float(mc_zooms[3]) if len(mc_zooms) >= 4 else 0.0
    spacing_xyz = tuple(float(s) for s in fixed_ref_nib.header.get_zooms()[:3])

    print(f"[SDC-4D] {prefix}: mc4d={mc4d_path.name} T={T} | "
          f"mov_geom={tuple(moving_ref_ants.shape)} --> fixed={tuple(fixed_img_ants.shape)}")

    # Apply inverse displacement per frame --> undistorted lattice:
    out_np = np.empty((*fixed_img_ants.shape, T), dtype=np.float32)
    for t in range(T):
        mov_t = ants.from_numpy(data[..., t].astype(np.float32))
        # Give the frame the distorted geometry so the displacement field applies correctly:
        mov_t = ants.copy_image_info(moving_ref_ants, mov_t)

        warped = ants.apply_transforms(
            fixed=fixed_img_ants,
            moving=mov_t,
            transformlist=[str(ax_warp)],
            interpolator="linear")

        out_np[..., t] = warped.numpy().astype(np.float32)

        if (t + 1) % max(1, T // 10) == 0 or (t == T - 1):
            print(f"    frame {t+1}/{T} done")

    # Scrub NaNs/Infs from SDC output BEFORE saving:
    out_np = np.nan_to_num(out_np, nan=0.0, posinf=0.0, neginf=0.0)

    # Save affine from undistorted ref, & TR from MC (& set units explicitly):
    out_img = nib.Nifti1Image(out_np, fixed_ref_nib.affine)
    out_img.header.set_xyzt_units('mm', 'sec')
    out_img.header.set_zooms(spacing_xyz + (TR,))
    nib.save(out_img, str(out_4d))

    print(f"[SDC-4D OK] {prefix}: wrote {out_4d}")
    return str(out_4d)


# ____________________________________________________________________________________________________________
# Main execution:

rows = prep_manifest.to_dict(orient="records")

out_list = []
for row in rows:
    try:
        p = _apply_sdc_to_known_mc4d(row, overwrite=OVERWRITE_SDC)
        out_list.append({"prefix": row["prefix"], "bold_mc_sdc": p})
    except Exception as e:
        print(f"[SDC-4D FAIL] {row.get('prefix','<unknown>')}: {e}")

if out_list:
    df = pd.DataFrame(out_list)
    with pd.option_context("display.max_colwidth", 160):
        display(df)
else:
    print("[SDC-4D] No runs processed.")

Final QC cell:

In [ ]:
# ==========================================================
# STEP 6 — QC (edge-only, single-color overlays):
#   * Shows ONLY yellow where EPI edges touch the T1 outline.
#   * BEFORE = magenta; AFTER = green
#   * 3 views (axial/coronal/sagittal), cropped to brain bounding-box
# ==========================================================

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def robust_window(x, mask=None, p_lo=2, p_hi=98):
    vals = x[mask] if (mask is not None and np.any(mask)) else x.ravel()
    lo, hi = np.percentile(vals, [p_lo, p_hi])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(vals)), float(max(np.max(vals), np.min(vals)+1.0))
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1)

def bbox_from_mask(mask_np, pad=0):
    if not np.any(mask_np):
        s = mask_np.shape
        return (slice(0,s[0]), slice(0,s[1]), slice(0,s[2]))
    xs, ys, zs = np.where(mask_np)
    x0, x1 = max(0, xs.min()-pad), min(mask_np.shape[0], xs.max()+1+pad)
    y0, y1 = max(0, ys.min()-pad), min(mask_np.shape[1], ys.max()+1+pad)
    z0, z1 = max(0, zs.min()-pad), min(mask_np.shape[2], zs.max()+1+pad)
    return (slice(x0,x1), slice(y0,y1), slice(z0,z1))

def mid_indices(mask_np):
    def mid_idx(arr, axis):
        inds = np.where(arr.any(axis=tuple(i for i in range(3) if i != axis)))[0]
        return (arr.shape[axis] // 2) if inds.size == 0 else int((inds.min()+inds.max())//2)
    return (mid_idx(mask_np,0), mid_idx(mask_np,1), mid_idx(mask_np,2))

def t1_inner_outline(mask3d):
    if not np.any(mask3d):
        return np.zeros_like(mask3d, dtype=bool)
    ero = binary_erosion(mask3d, iterations=1)
    return mask3d & (~ero)  # 1-pixel inner border

def epi_edges_2d(epi2d, restrict2d, sigma=1.2):
    # Compute EPI edges (restricts statistics to band near outline to avoid background driving thresholds):
    if np.any(restrict2d):
        vals = epi2d[restrict2d]
        lo, hi = np.percentile(vals, [2, 98])
        epi_n = np.clip((epi2d - lo) / (hi - lo + 1e-8), 0, 1)
    else:
        epi_n = robust_window(epi2d, None, *QC_INT_PLOTLIMS)

    if HAVE_SK:
        return sk_canny(epi_n, sigma=sigma)
    # Fallback -- gradient magnitude + high-percentile threshold:
    gx, gy = np.gradient(gaussian_filter(epi_n, sigma=max(1.0, sigma)))
    gm = np.hypot(gx, gy)
    thr = np.percentile(gm[restrict2d] if np.any(restrict2d) else gm, 90)
    return gm >= thr

def overlay_edge_only(t1_3d, epi_3d, brain3d, outline3d, plane, idx, epi_tint):
    # Return RGB for one slice; grayscale T1 + epi_tint, with ONLY yellow where EPI edges touch outline:
    if plane == 'ax':
        t1, epi, brain, outline = t1_3d[:,:,idx], epi_3d[:,:,idx], brain3d[:,:,idx], outline3d[:,:,idx]
    elif plane == 'co':
        t1, epi, brain, outline = t1_3d[:,idx,:], epi_3d[:,idx,:], brain3d[:,idx,:], outline3d[:,idx,:]
    elif plane == 'sa':
        t1, epi, brain, outline = t1_3d[idx,:,:], epi_3d[idx,:,:], brain3d[idx,:,:], outline3d[idx,:,:]
    else:
        raise ValueError("plane must be 'ax','co','sa'")

    # base grayscale + EPI tint:
    t1w = robust_window(t1, brain, *QC_INT_PLOTLIMS)
    epiw = robust_window(epi, brain, *QC_INT_PLOTLIMS)
    rgb  = np.stack([t1w, t1w, t1w], axis=-1)
    tint = np.zeros_like(rgb)
    tint[...,0] = epi_tint[0] * epiw
    tint[...,1] = epi_tint[1] * epiw
    tint[...,2] = epi_tint[2] * epiw
    a    = (QC_BLEND_ALPHA * brain).astype(float)[..., None]
    rgb  = (1.0 - a) * rgb + a * tint

    # Restrict edges to a small band around the outline:
    band = binary_dilation(outline, iterations=max(1, QC_OUTLINE_TOUCH_R))
    edges = epi_edges_2d(epi, band, sigma=QC_CANNY_SIGMA) & band

    # keep only the pixels where edges actually hit the outline:
    yellow = edges & outline
    y = yellow.astype(bool)
    if np.any(y):
        rgb[y] = (1.0, 1.0, 0.0)

    # return as (y,x,3) for 'imshow' function:
    return np.transpose(rgb, (1,0,2))

def edge_touch_fraction(epi2d, outline2d, band_radius=1, sigma=QC_CANNY_SIGMA):
    """
    Fraction of outline pixels that have an EPI edge within <= band_radius pixels.
    More forgiving than exact overlap; better reflects small, desirable inward shifts.
    """
    # Detect edges:
    band = binary_dilation(outline2d, iterations=max(1, band_radius))
    if HAVE_SK:
        vals = epi2d[band]
        lo, hi = np.percentile(vals, [2, 98]) if vals.size else (0.0, 1.0)
        epiw = np.clip((epi2d - lo) / (hi - 1e-8 - lo), 0, 1)
        edges = sk_canny(epiw, sigma=sigma)
    else:
        epiw = robust_window(epi2d, band, *QC_INT_PLOTLIMS)
        gx, gy = np.gradient(gaussian_filter(epiw, sigma=max(1.0, sigma)))
        gm = np.hypot(gx, gy)
        thr = np.percentile(gm[band] if np.any(band) else gm, 90)
        edges = gm >= thr

    # Distance-to-edge image:
    inv_edges = ~edges
    dist_to_edge = _dt(inv_edges)  # <-- '0' where edges==True; grows with distance

    # Outline pixels counted as "hit" if within band_radius:
    outline_idx = outline2d.astype(bool)
    hits_mask = np.zeros_like(outline2d, dtype=bool)
    hits_mask[outline_idx] = (dist_to_edge[outline_idx] <= float(band_radius))

    hits = int(hits_mask.sum())
    denom = int(outline_idx.sum())
    return float(hits) / max(denom, 1), hits, denom

def slice_touch_scores(t1_np, epi_np, outline, brain_m, mx, my, mz):
    scores = {}
    # axial (z=mz):
    scores['ax'] = edge_touch_fraction(epi_np[:,:,mz], outline[:,:,mz],
                                       band_radius=QC_OUTLINE_TOUCH_R, sigma=QC_CANNY_SIGMA)
    # coronal (y=my):
    scores['co'] = edge_touch_fraction(epi_np[:,my,:],  outline[:,my,:],
                                       band_radius=QC_OUTLINE_TOUCH_R, sigma=QC_CANNY_SIGMA)
    # sagittal (x=mx):
    scores['sa'] = edge_touch_fraction(epi_np[mx,:,:],  outline[mx,:,:],
                                       band_radius=QC_OUTLINE_TOUCH_R, sigma=QC_CANNY_SIGMA)
    return scores  # each value is (fraction, hits, outline_px)


# ________________________________________________________________________________________
# Main execution:

rows = prep_manifest.to_dict(orient="records")

for row in rows:
    prefix       = row["prefix"]
    boldref_path = row["boldref_path"]                # <-- 'BEFORE' image
    mask_path    = row["mask_path"]                   # <-- EPI brain mask
    t1_path      = row["t1_brain_nii"]
    t1mask_path  = row["t1_mask_nii"]
    out_sdc      = Path(SDC_OUTPUT_DIR) / prefix / "boldref_sdc.nii"

    if not out_sdc.exists():
        print(f"[QC SKIP] {prefix}: missing boldref_sdc.nii (run Step 4 apply first)")
        continue

    ensure_dir(QC_DIR)

    png_before = Path(QC_DIR) / f"{prefix}_04a_before_3view_singleline.png"
    png_after  = Path(QC_DIR) / f"{prefix}_04b_after_3view_singleline.png"

    # Load & resample T1 to EPI grid:
    epi_b  = ants.image_read(boldref_path)
    epi_a  = ants.image_read(str(out_sdc))
    emask  = ants.image_read(mask_path)
    t1     = ants.image_read(t1_path)
    t1mask = ants.image_read(t1mask_path)

    t1_e   = ants.resample_image_to_target(t1,     epi_b, interp_type=0)
    t1m_e  = ants.resample_image_to_target(t1mask, epi_b, interp_type=1)

    epi_b_np = epi_b.numpy().astype(np.float32)
    epi_a_np = epi_a.numpy().astype(np.float32)
    epi_m_np = (emask.numpy()  > 0)
    t1_np    = t1_e.numpy().astype(np.float32)
    t1m_np   = (t1m_e.numpy()  > 0)

    brain_m  = epi_m_np & t1m_np

    # Crop to brain bounding-box:
    sl = bbox_from_mask(brain_m, pad=QC_CROP_PAD)
    epi_b_np = epi_b_np[sl]; epi_a_np = epi_a_np[sl]
    epi_m_np = epi_m_np[sl]; t1_np    = t1_np[sl]
    t1m_np   = t1m_np[sl];   brain_m  = brain_m[sl]

    outline = t1_inner_outline(t1m_np)
    mx, my, mz = mid_indices(brain_m)

    # Generate "BEFORE" image (magenta):
    img_ax_b = overlay_edge_only(t1_np, epi_b_np, brain_m, outline, 'ax', mz, (1,0,1))
    img_co_b = overlay_edge_only(t1_np, epi_b_np, brain_m, outline, 'co', my, (1,0,1))
    img_sa_b = overlay_edge_only(t1_np, epi_b_np, brain_m, outline, 'sa', mx, (1,0,1))
    fig, axs = plt.subplots(1, 3, figsize=FIGSIZE)
    for ax, img, ttl in zip(
        axs,
        [img_ax_b, img_co_b, img_sa_b],
        [f"Axial z={mz}", f"Coronal y={my}", f"Sagittal x={mx}"]):
        ax.imshow(img, origin='lower', interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(ttl, fontsize=11)
    fig.suptitle(f"{prefix} — BEFORE (magenta) | edge-only overlay (yellow = EPI edge on T1 outline)", fontsize=12)
    plt.tight_layout(rect=[0,0,1,0.93]); plt.savefig(png_before, dpi=DPI); plt.close(fig)

    # Generate "AFTER" image (green):
    img_ax_a = overlay_edge_only(t1_np, epi_a_np, brain_m, outline, 'ax', mz, (0,1,0))
    img_co_a = overlay_edge_only(t1_np, epi_a_np, brain_m, outline, 'co', my, (0,1,0))
    img_sa_a = overlay_edge_only(t1_np, epi_a_np, brain_m, outline, 'sa', mx, (0,1,0))
    fig, axs = plt.subplots(1, 3, figsize=FIGSIZE)
    for ax, img, ttl in zip(
        axs,
        [img_ax_a, img_co_a, img_sa_a],
        [f"Axial z={mz}", f"Coronal y={my}", f"Sagittal x={mx}"]):
        ax.imshow(img, origin='lower', interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(ttl, fontsize=11)
    fig.suptitle(f"{prefix} — AFTER (green) | edge-only overlay (yellow = EPI edge on T1 outline)", fontsize=12)
    plt.tight_layout(rect=[0,0,1,0.93]); plt.savefig(png_after, dpi=DPI); plt.close(fig)

    # Numeric QC (edge-touch fractions; same slices as used in plots):
    scores_before = slice_touch_scores(t1_np, epi_b_np, outline, brain_m, mx, my, mz)
    scores_after  = slice_touch_scores(t1_np, epi_a_np, outline, brain_m, mx, my, mz)

    def _fmt_scores(sc):
        # sc[view] -> (frac, hits, denom):
        return " | ".join([
            f"{v.upper()}={sc[v][0]*100:.1f}% ({sc[v][1]}/{sc[v][2]})"
            for v in ("ax","co","sa")])

    print(f"[QC metric] {prefix}: edge-touch BEFORE  :: {_fmt_scores(scores_before)}")
    print(f"[QC metric] {prefix}: edge-touch AFTER   :: {_fmt_scores(scores_after)}")

    # Optional 4D sanity note (no plot changes):
    mc4d  = Path(FULL_MC_VOLUME_DIR) / f"{prefix}_bold_mc.nii"
    sdc4d = Path(SDC_OUTPUT_DIR) / prefix / "bold_mc_sdc.nii.gz"
    if mc4d.exists() and sdc4d.exists():
        try:
            mc_hdr  = nib.load(str(mc4d)).header
            sdc_img = nib.load(str(sdc4d))
            sdc_hdr = sdc_img.header
            tr_mc   = mc_hdr.get_zooms()[3] if len(mc_hdr.get_zooms()) >= 4 else float("nan")
            tr_sdc  = sdc_hdr.get_zooms()[3] if len(sdc_hdr.get_zooms()) >= 4 else float("nan")
            shape_sdc = sdc_img.shape
            print(f"[QC 4D] {prefix}: bold_mc_sdc shape={shape_sdc} | TR(mc)={tr_mc:.4f}s | TR(sdc4d)={tr_sdc:.4f}s")
        except Exception as e:
            print(f"[QC 4D WARN] {prefix}: could not read 4D SDC header: {e}")

    print(f"[QC edge-only] {prefix}:\n  BEFORE: {png_before}\n  AFTER : {png_after}")

---------